In [8]:
from IMH import IndMH
from estimators import Delta, Delta_con, Gamma, Gamma_con, Gamma_con_Lap
import numpy.random as nr
import numpy as np
import haiku as hk
import jax


In [12]:
epoch = 100
Itr = 10000
N = 4096
d = 16
K = 40
S0 = 50
r = 0.05
v = 0.1
T = 1
distr = "multivariate_normal"
objective = "Gamma"
conditional = False
ifLap = False

In [13]:
cmc_str = "F_CMC_" + objective
if conditional:
    obj = objective + '_con'
    if ifLap:
        obj = obj + '_Lap'
        d = d - 1
    print("===", obj, "Pricing problem = Independent Metropolis Hastings + control variate + conditional method ===")
else:
    obj = objective
    print("===", obj, "Pricing problem = Independent Metropolis Hastings + control variate ===")

est = eval(obj)(K, S0, r, v, T, d)
est.Generative_Matrix()
est.Lognormal_Approx()
if ifLap:
    est.Lap()
    mu1 = est.Lap_tar_mean
    L1 = est.Lap_tar_L
    mu2 = est.Lap_pro_mean
    L2 = est.Lap_pro_L
else:
    mu1 = est.default_mean
    L1 = est.default_L
    mu2 = est.proposal_mean
    L2 = est.proposal_L
MyIMH = IndMH(distr, distr, Itr, N)
MyIMH.set_distribution_parameters(mu1, L1, mu2, L2)
MyIMH.init_state()
print("Initialization completed !")

=== Gamma Pricing problem = Independent Metropolis Hastings + control variate ===
Initialization completed !


In [14]:

epoch = 10
cmc_mean = []
imhcv_mean = []
for i in range(epoch):

    datax, datay, accept = MyIMH.IMH() 
    cmcdata = eval('nr.' + distr)(est.default_mean, est.default_L @ est.default_L.T, N)
    imcv = est.IMCV(datax, datay, accept)
    cmc = getattr(est, cmc_str)(cmcdata)
    imhcv_mean.append(imcv[0])
    cmc_mean.append(cmc[0])
    if (i + 1) % 10 == 0:
        print(i + 1, "replications completed !")

cmc_mean = np.array(cmc_mean)
imhcv_mean = np.array(imhcv_mean)
VRF = cmc_mean.var()/imhcv_mean.var()

10 replications completed !


In [15]:
nl = len(str(round(VRF)))
print("=" * 10, "Compare with CMC", "=" * 10)
print(f"|    Mean of CMC    =  {cmc_mean.mean():.6f}      |")
print(f"|    Mean of IMHCV  =  {imhcv_mean.mean():.6f}      |")
print(f"|    Var of CMC     =  {cmc_mean.var():.5e}   |")
print(f"|    Var of IMHCV   =  {imhcv_mean.var():.5e}   |")
print("|    V R F          = ", round(VRF, 2), " " * (9 - nl), "|")
print('=' * 38)

========== Compare with CMC ==========
|    Mean of CMC    =  -0.003526      |
|    Mean of IMHCV  =  0.000032      |
|    Var of CMC     =  1.18063e-04   |
|    Var of IMHCV   =  9.89383e-07   |
|    V R F          =  119.33        |
